In [12]:
import pickle
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


# Load the data

In [13]:
# ========= X_train ========== #
X_train = pd.read_csv("../data/training/X_train.csv").drop(["Unnamed: 0"], axis = 1)
y_train = pd.read_csv("../data/training/y_train.csv").drop(["Unnamed: 0"], axis = 1)

# ========== X_test ========== #
X_val = pd.read_csv("../data/training/X_val.csv").drop(["Unnamed: 0"], axis = 1)
y_val = pd.read_csv("../data/training/y_val.csv").drop(["Unnamed: 0"], axis = 1)

print(f"Shape fo X_train : {X_train.shape}")
print(f"Shape fo y_train : {y_train.shape}")
print(f"Shape fo X_test : {X_val.shape}")
print(f"Shape fo y_test : {y_val.shape}")

Shape fo X_train : (506, 2053)
Shape fo y_train : (506, 1)
Shape fo X_test : (127, 2053)
Shape fo y_test : (127, 1)


# Label Encoder 

In [14]:
def encode_labels(y):
    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y)
    return y_encoded, encoder

y_train_encoded, encoder = encode_labels(y_train)
print(f"Shape of y_train : {y_train.shape}")

Shape of y_train : (506, 1)


/Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/PhysicoNet/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [15]:
# Dictionary to store classifiers
CLASSIFIERS = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVC": SVC(kernel='linear', probability=True),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "NaiveBayes": GaussianNB()
}

def train_model(X_train, y_train, model_name):
    """
    Train the specified classifier
    """
    if model_name not in CLASSIFIERS:
        raise ValueError(f"Model {model_name} not recognized. Choose from {list(CLASSIFIERS.keys())}")
    
    model = CLASSIFIERS[model_name]
    model.fit(X_train, y_train)
    return model

def evaluate_model(model, X_test, y_test):
    """
    Evaluate the model on test data and return all classification metrics
    """
    y_pred = model.predict(X_test)
    
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
        "recall": recall_score(y_test, y_pred, average='weighted', zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average='weighted', zero_division=0),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "classification_report": classification_report(y_test, y_pred, zero_division=0)
    }
    
    return metrics


In [16]:
channel_info = pd.read_csv("../data/analysis/ChannelInformationInfo.csv")

region_of_interest = {
    "Region1": ["Fc1.", "Fc2.", "Fc3.", "Fc4.", "Fc5.", "Fc6."], 
    "Region2": ["C5..", "C6..", "C3..", "C4..","C1..", "C2.."],
    "Region3": ["Cp1.", "Cp2.", "Cp3.", "Cp4.", "Cp5.", "Cp6."], 
    "Region4": ["Fc3.", "Fc4.", "C5..", "C6..", "C3..", "C4..", "C1..", "C2..", "Cp3.", "Cp4."], 
    "Region5": ["Fc1.", "Fc2.", "Fc3.", "Fc4.", "Cp1.", "Cp2.", "Cp3.", "Cp4.", "C1..", "C2..", "C3..", "C4.."]
}

roi_columns = channel_info[channel_info["Region"].isin(region_of_interest["Region1"])]["ColumnNames"].to_list()

X_train_roi  = X_train[roi_columns]
X_val_roi  = X_val[roi_columns]

#### Step 1 : RandomForest

In [17]:
model = train_model(X_train_roi, y_train, "RandomForest")
metrics = evaluate_model(model, X_val_roi, y_val)

/Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/PhysicoNet/.venv/lib/python3.12/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [18]:
len(model.feature_importances_)

192

In [19]:
print(metrics["classification_report"])

              precision    recall  f1-score   support

          T1       0.54      0.56      0.55        64
          T2       0.53      0.51      0.52        63

    accuracy                           0.54       127
   macro avg       0.54      0.54      0.53       127
weighted avg       0.54      0.54      0.54       127



#### Step 2: Naive Bayes

In [20]:
model = train_model(X_train_roi, y_train, "NaiveBayes")
metrics = evaluate_model(model, X_val_roi, y_val)

/Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/PhysicoNet/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [21]:
metrics

{'accuracy': 0.5669291338582677,
 'precision': 0.5745918955252545,
 'recall': 0.5669291338582677,
 'f1_score': 0.5575580338126827,
 'confusion_matrix': array([[27, 37],
        [18, 45]]),
 'classification_report': '              precision    recall  f1-score   support\n\n          T1       0.60      0.42      0.50        64\n          T2       0.55      0.71      0.62        63\n\n    accuracy                           0.57       127\n   macro avg       0.57      0.57      0.56       127\nweighted avg       0.57      0.57      0.56       127\n'}

In [22]:
print(metrics["classification_report"])

              precision    recall  f1-score   support

          T1       0.60      0.42      0.50        64
          T2       0.55      0.71      0.62        63

    accuracy                           0.57       127
   macro avg       0.57      0.57      0.56       127
weighted avg       0.57      0.57      0.56       127



#### Step 3 : KNearestNeighbors

In [23]:
model = train_model(X_train_roi, y_train, "KNN")
metrics = evaluate_model(model, X_val_roi, y_val)

/Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/PhysicoNet/.venv/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


In [24]:
metrics

{'accuracy': 0.41732283464566927,
 'precision': 0.41447196713873286,
 'recall': 0.41732283464566927,
 'f1_score': 0.41331893108563833,
 'confusion_matrix': array([[32, 32],
        [42, 21]]),
 'classification_report': '              precision    recall  f1-score   support\n\n          T1       0.43      0.50      0.46        64\n          T2       0.40      0.33      0.36        63\n\n    accuracy                           0.42       127\n   macro avg       0.41      0.42      0.41       127\nweighted avg       0.41      0.42      0.41       127\n'}

In [25]:
print(metrics["classification_report"])

              precision    recall  f1-score   support

          T1       0.43      0.50      0.46        64
          T2       0.40      0.33      0.36        63

    accuracy                           0.42       127
   macro avg       0.41      0.42      0.41       127
weighted avg       0.41      0.42      0.41       127

